项目介绍：本项目主要侧重于CRAG的实现和评估，用Transformer经典文献来检测搜索效果，链接：https://arxiv.org/pdf/1706.03762
 ———— CRAG：先搜索文献内容，如果无法搜索到或搜索质量较差，则联网搜索再给出答案
 ———— 评估方法：ragas，包括faithfulness、Answer Relevance、、Context Precision、Context Recall

In [18]:
import os
from dotenv import load_dotenv
import dashscope

load_dotenv()
qwen_api_key = os.getenv("DASHSCOPE_API_KEY")
# 输入通义千问 API Key（从阿里云百炼平台获取：https://dashscope.console.aliyun.com/）
if not qwen_api_key:
    raise ValueError("未找到 DASHSCOPE_API_KEY，请检查 .env 文件！")

dashscope.api_key = qwen_api_key
print("API Key 加载成功，准备就绪！")

API Key 加载成功，准备就绪！


In [19]:
#import bs4
import os
from openai import OpenAI
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.embeddings import Embeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# 自定义通义千问 Embedding 类（完善分批处理和错误防护）
class QwenEmbeddings(Embeddings):
    def __init__(self, model="text-embedding-v4", api_key=None, base_url=None, batch_size=10):
        self.model = model
        self.batch_size = batch_size  # 通义千问API强制限制最大10条/批
        # 初始化OpenAI客户端
        self.client = OpenAI(
            api_key=dashscope.api_key or os.getenv("DASHSCOPE_API_KEY"),
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
        )

    def embed_documents(self, texts):
        """批量嵌入文档 - 严格分批处理，过滤空文本"""
        # 过滤空文本（避免API报错）
        texts = [text.strip() for text in texts if text.strip()]
        if not texts:
            return []
        
        embeddings = []
        # 分批处理：每批不超过batch_size条
        for i in range(0, len(texts), self.batch_size):
            batch_texts = texts[i:i + self.batch_size]
            
            # 调用API
            completion = self.client.embeddings.create(
                model=self.model,
                input=batch_texts
            )
            
            # 提取embedding并保持顺序一致
            batch_embeddings = [item.embedding for item in completion.data]
            embeddings.extend(batch_embeddings)
        
        return embeddings
    
    def embed_query(self, text):
        """嵌入查询文本 - 处理单个文本"""
        text = text.strip()
        if not text:
            return []
        
        completion = self.client.embeddings.create(
            model=self.model,
            input=text
        )
        return completion.data[0].embedding

In [20]:
#PDF文档加载器
import os
import requests
import tempfile
from langchain_community.document_loaders import UnstructuredPDFLoader, WebBaseLoader
from langchain_community.document_loaders import PyMuPDFLoader

class PDFLoader(WebBaseLoader):
    def __init__(self, file_path):
        super().__init__(web_path=file_path)
        self.url = file_path

    def load(self):
        """
        利用 Unstructured 架构进行深度解析：可以解析图片、表格、标题等不同元素，并在元数据中保留这些信息
        """
        try:
            print(f"正在启动 PyMuPDF 解析任务: {self.url}")
            response = requests.get(self.url, timeout=15)
            response.raise_for_status()

            temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
            temp_file.write(response.content)
            temp_path = temp_file.name
            temp_file.close()  # 重要：写入后立即关闭，释放文件锁

            try:
                # 2. 现在 PyMuPDF 可以自由访问这个文件了
                loader = PyMuPDFLoader(temp_path)
                docs = loader.load()
                print(f"解析完成！共提取 {len(docs)} 页内容。")
            finally:
                # 3. 无论解析是否成功，最后都把临时文件删掉，保持电脑干净
                if os.path.exists(temp_path):
                    os.remove(temp_path)

            return docs

        except Exception as e:
            print(f"PyMuPDF 解析失败: {e}")
            return []
        
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
loader = PDFLoader(pdf_url)
docs = loader.load()

if docs:
    print(f"\n--- 文献解析概览 ---")
    print(f"片段总数: {len(docs)}")
        
    for i, doc in enumerate(docs[:3]):
        category = doc.metadata.get("category")
        content_preview = doc.page_content[:50].replace("\n", " ")
        print(f"片段 {i} | 类型: {category} | 内容: {content_preview}...")

正在启动 PyMuPDF 解析任务: https://arxiv.org/pdf/1706.03762.pdf
解析完成！共提取 15 页内容。

--- 文献解析概览 ---
片段总数: 15
片段 0 | 类型: None | 内容: Provided proper attribution is provided, Google he...
片段 1 | 类型: None | 内容: 1 Introduction Recurrent neural networks, long sho...
片段 2 | 类型: None | 内容: Figure 1: The Transformer - model architecture. Th...


In [21]:
import re

def clean_document_content(docs):
    """
    在分割前，对原始页面的文字进行手术级清洗
    """
    cleaned_docs = []
    for doc in docs:
        content = doc.page_content
        # 1. 过滤掉页码 (如 "Page 1 of 12" 或 单独的数字)
        content = re.sub(r'Page \d+ of \d+', '', content)
        # 2. 过滤掉常见的论文页眉干扰
        content = re.sub(r'arXiv:\d+\.\d+v\d+ \[cs\.CL\] \d+ \w+ \d+', '', content)
        # 3. 修复连字符带来的断词
        content = content.replace("-\n", "")
        # 4. 将多个换行符替换为标准分段，去除多余空格
        content = re.sub(r'\n+', '\n', content)
        content = content.strip()
        
        # 只保留有实质内容的页面
        if len(content) > 10:
            doc.page_content = content
            cleaned_docs.append(doc)
            
    print(f"清洗完成：已处理 {len(cleaned_docs)} 页内容")
    return cleaned_docs

cleaned_docs = clean_document_content(docs)

清洗完成：已处理 15 页内容


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_text_chunks(docs, chunk_size=600, chunk_overlap=60):
    """
    函数式写法：输入原始文档，返回切割好的块
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "。", "！", "？", " ", ""]
    )
    
    chunks = splitter.split_documents(docs)
    print(f"已切分为 {len(chunks)} 个片段")
    return chunks

chunks = get_text_chunks(cleaned_docs, chunk_size=600, chunk_overlap=60)

已切分为 78 个片段


In [23]:

# Embed - 确保使用带分批处理的QwenEmbeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=QwenEmbeddings(
        model="text-embedding-v4",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        batch_size=10  # 明确指定批量大小，与API限制一致
    )
)

retriever = vectorstore.as_retriever()

In [24]:
# 生成答案
from click import prompt
from langchain.chains import RetrievalQA
from langchain_community.llms import Tongyi  # 或者使用你的 OpenAI 适配类
from langchain.prompts import PromptTemplate
from typer import prompt

def get_answer_from_pdf(query, retriever):
    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template="""你是一个严谨的学术助手。
        请严格根据下方提供的【参考资料】来回答用户的问题。
        要求：
        1. 只能使用资料中的事实，严禁发挥。
        2. 如果资料未提及，请回答“资料中未找到相关内容”。
        3. 涉及计算复杂度或具体数值（如 d_model）时，必须完全与原文一致
        参考资料：{context}
        待回答的问题：{question}
        请用专业、简洁的中文回答："""
    )

    llm = ChatOpenAI(
        model_name="qwen-turbo",
        temperature=0,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        max_tokens=1024
    )
    # Post-processing
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # Chain: Retriever -> Format -> Prompt -> LLM -> Output Parser
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    answer = rag_chain.invoke(query)
    return answer   

query = input("请输入你的问题：")
answer = get_answer_from_pdf(query, retriever)
print("答案：", answer)

答案： Transformer 的核心创新是作为首个完全依赖自注意力机制来计算输入和输出表示的转换模型，而不使用序列对齐的 RNN 或卷积。


In [28]:
# Ragas 评测：与 advanced_rag.ipynb 使用相同的四项指标
import os
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from langchain_openai import ChatOpenAI
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextRecall,
    LLMContextPrecisionWithReference,
)

# 导入统一的评测集定义
from evaluation_dataset import eval_questions_50, eval_ground_truths_50

# 1. 定义"裁判员" LLM
evaluator_llm = ChatOpenAI(
    model_name="qwen-max",
    api_key=os.environ["DASHSCOPE_API_KEY"],
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0,
    max_tokens=1024,
)

# 2. 定义 Embeddings
evaluator_embeddings = QwenEmbeddings(
    model="text-embedding-v4",
    api_key=os.environ["DASHSCOPE_API_KEY"],
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    batch_size=10,
)

# 3. 实例化指标对象
metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerRelevancy(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    LLMContextPrecisionWithReference(llm=evaluator_llm),
]

def answer_question_for_eval(main_question: str):
    """运行 baseline RAG 流程，并返回答案与检索上下文"""
    docs = retriever.invoke(main_question)
    docs = [doc for doc in docs if getattr(doc, "page_content", "").strip()]
    response = get_answer_from_pdf(main_question, retriever)
    retrieved_contexts = [doc.page_content for doc in docs]
    return response, retrieved_contexts

def build_ragas_dataset(eval_questions: list, eval_ground_truths: list, limit: int | None = None):
    """构造 baseline 的 Ragas 评测数据集；limit 可用于小样本调试"""
    total = len(eval_questions) if limit is None else min(limit, len(eval_questions))
    records = []

    for idx, (user_input, reference) in enumerate(zip(eval_questions[:total], eval_ground_truths[:total]), start=1):
        print(f"[{idx}/{total}] 正在生成 baseline 回答与上下文: {user_input}")
        response, retrieved_contexts = answer_question_for_eval(user_input)
        records.append({
            "user_input": user_input,
            "response": response,
            "reference": reference,
            "retrieved_contexts": retrieved_contexts,
        })

    dataset = Dataset.from_list(records)
    return dataset, pd.DataFrame(records)

def run_ragas_evaluation(limit: int | None = None, evaluator_model_name: str = "qwen-max"):
    """执行与 advanced notebook 一致的四项 Ragas 指标评测"""
    eval_dataset, eval_records_df = build_ragas_dataset(
        eval_questions_50,
        eval_ground_truths_50,
        limit=limit,
    )

    print("🧪 Ragas 正在进行语义对齐与逻辑打分...")
    eval_result = evaluate(
        dataset=eval_dataset,
        metrics=metrics,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        raise_exceptions=False,
        show_progress=True,
    )

    score_df = eval_result.to_pandas()
    metric_cols = [
        col
        for col in [
            "faithfulness",
            "answer_relevancy",
            "context_recall",
            "llm_context_precision_with_reference",
        ]
        if col in score_df.columns
    ]
    merged_df = pd.concat([eval_records_df.reset_index(drop=True), score_df[metric_cols].reset_index(drop=True)], axis=1)
    return merged_df

C:\Users\59118\AppData\Local\Temp\ipykernel_54596\4062864981.py:7: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\59118\AppData\Local\Temp\ipykernel_54596\4062864981.py:7: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\59118\AppData\Local\Temp\ipykernel_54596\4062864981.py:7: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\59118\AppData\Local\Temp\ipykernel_54596\

In [29]:
# 验证评测集已加载并运行小样本评测
assert len(eval_questions_50) == 50
assert len(eval_ground_truths_50) == 50
print(f"✅ 已加载 {len(eval_questions_50)} 个问题和 {len(eval_ground_truths_50)} 个标准答案。（包含5个陷阱题/推理题，占比10%）")

# 先小样本调试（推荐）
sample_report = run_ragas_evaluation(limit=5)

# 全量 50 题正式评测
# full_report = run_ragas_evaluation()

# 查看结果时可执行：
display(sample_report.head())
# display(full_report.head())

✅ 已加载 50 个问题和 50 个标准答案。（包含5个陷阱题/推理题，占比10%）
[1/5] 正在生成 baseline 回答与上下文: Transformer 模型采用了什么总体架构？
[2/5] 正在生成 baseline 回答与上下文: 原始 Transformer 中编码器和解码器各堆叠了多少层？
[3/5] 正在生成 baseline 回答与上下文: 编码器每一层包含哪两个主要子层？
[4/5] 正在生成 baseline 回答与上下文: 解码器每一层相比编码器额外多了什么子层？
[5/5] 正在生成 baseline 回答与上下文: Transformer 在每个子层外部使用了哪两种稳定训练的结构？
🧪 Ragas 正在进行语义对齐与逻辑打分...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:   5%|▌         | 1/20 [00:04<01:30,  4.78s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating:  35%|███▌      | 7/20 [00:16<00:23,  1.85s/it]d:\project\CRAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:406: UserWarning: Unexpected type for token usage: <class 'NoneType'>
  warnings.warn(f"Unexpected type for token usage: {type(new_usage)}")
Evaluating: 100%|██████████| 20/20 [00:33<00:00,  1.65s/it]


,user_input,response,reference,retrieved_contexts,faithfulness,answer_relevancy,context_recall,llm_context_precision_with_reference
0,Transformer 模型采用了什么总体架构？,Transformer 模型采用的总体架构是使用堆叠的自注意力机制和点对点的全连接层，分别用...,Transformer 采用标准的编码器-解码器架构，编码器和解码器都由堆叠的层组成。,[Figure 1: The Transformer - model architectur...,1.000000,0.774531,1.0,1.000000
1,原始 Transformer 中编码器和解码器各堆叠了多少层？,原始 Transformer 中编码器和解码器各堆叠了 6 层。,原始 Transformer 的 base 配置使用 6 层编码器和 6 层解码器。,[Figure 1: The Transformer - model architectur...,1.000000,0.889297,1.0,1.000000
2,编码器每一层包含哪两个主要子层？,编码器每一层包含的两个主要子层是多头自注意力机制和简单的位置全连接前馈网络。,编码器每层包含多头自注意力子层和位置前馈全连接网络子层。,"[the two sub-layers, followed by layer normali...",1.000000,0.914438,1.0,0.416667
3,解码器每一层相比编码器额外多了什么子层？,解码器每一层相比编码器额外多了第三个子层，该子层执行多头注意力机制，对编码器堆栈的输出进行多...,解码器每层比编码器多了一个对编码器输出执行注意力的编码器-解码器注意力子层。,"[the two sub-layers, followed by layer normali...",1.000000,0.975559,1.0,1.000000
4,Transformer 在每个子层外部使用了哪两种稳定训练的结构？,Transformer 在每个子层外部使用了残差连接和层归一化结构。,每个子层外部都使用残差连接，并在残差相加后进行层归一化。,[1.1 · 1021\nConvS2S Ensemble [9]\n26.36\n41.2...,0.666667,0.894525,1.0,1.000000


In [30]:
metric_cols = [
    col
    for col in [
        "faithfulness",
        "answer_relevancy",
        "llm_context_recall",
        "llm_context_precision_with_reference",
    ]
    if col in sample_report.columns
 ]

print("Metric means:")
print(sample_report[metric_cols].mean(numeric_only=True))

diag_cols = [col for col in [
    "user_input",
    "reference",
    "response",
    "faithfulness",
    "answer_relevancy",
    "llm_context_recall",
    "llm_context_precision_with_reference",
 ] if col in sample_report.columns]

display(sample_report[diag_cols].head(5))

Metric means:
faithfulness                            0.933333
answer_relevancy                        0.889670
llm_context_precision_with_reference    0.883333
dtype: float64


,user_input,reference,response,faithfulness,answer_relevancy,llm_context_precision_with_reference
0,Transformer 模型采用了什么总体架构？,Transformer 采用标准的编码器-解码器架构，编码器和解码器都由堆叠的层组成。,Transformer 模型采用的总体架构是使用堆叠的自注意力机制和点对点的全连接层，分别用...,1.000000,0.774531,1.000000
1,原始 Transformer 中编码器和解码器各堆叠了多少层？,原始 Transformer 的 base 配置使用 6 层编码器和 6 层解码器。,原始 Transformer 中编码器和解码器各堆叠了 6 层。,1.000000,0.889297,1.000000
2,编码器每一层包含哪两个主要子层？,编码器每层包含多头自注意力子层和位置前馈全连接网络子层。,编码器每一层包含的两个主要子层是多头自注意力机制和简单的位置全连接前馈网络。,1.000000,0.914438,0.416667
3,解码器每一层相比编码器额外多了什么子层？,解码器每层比编码器多了一个对编码器输出执行注意力的编码器-解码器注意力子层。,解码器每一层相比编码器额外多了第三个子层，该子层执行多头注意力机制，对编码器堆栈的输出进行多...,1.000000,0.975559,1.000000
4,Transformer 在每个子层外部使用了哪两种稳定训练的结构？,每个子层外部都使用残差连接，并在残差相加后进行层归一化。,Transformer 在每个子层外部使用了残差连接和层归一化结构。,0.666667,0.894525,1.000000
